# Modelo de Popularidad

En este notebook se presenta el entrenamiento de un modelo cuyo objetivo es predecir la popularidad de canciones. Por lo cual, el notebook se divide en las siguientes secciones:

1. Importacion de librerias necesarias.
2. Cargue de datos.
3. Categorización de la variable objetivo.
4. Búsqueda del mejor modelo y sus hiperparámetros. 
5. Guardado del mejor modelo.

## 1. Importación de las librerias necesarias


In [0]:
import pandas as pd
import os
import warnings
import yaml

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import mlflow
import mlflow.sklearn

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV
import matplotlib.pyplot as plt

import mlflow.xgboost
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
import numpy as np
# Suprimir los warnings para una salida más limpia
warnings.filterwarnings('ignore')

# Opciones para una mejor visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("¡Bibliotecas importadas con éxito!")

¡Bibliotecas importadas con éxito!


## Cargue de datos
Se realiza el cargue de los datos que ya fueron preparados

In [0]:
# Define the path to the dataset
config = yaml.safe_load(open("params.yaml"))["prepare"]

output_path_train=config['output_path_train']
output_path_test=config['output_path_test']

X_train_path = os.path.join(output_path_train, 'X_train.csv')
X_test_path = os.path.join(output_path_test, 'X_test.csv')
y_train_path = os.path.join(output_path_train, 'y_train.csv')
y_test_path = os.path.join(output_path_test, 'y_test.csv')


# Load the dataset
try:
    X_train = pd.read_csv(X_train_path)
    X_test = pd.read_csv(X_test_path)
    y_train = pd.read_csv(y_train_path)
    y_test = pd.read_csv(y_test_path)
    print(f"\n✓ Datasets loaded successfully!")
    print(f"Datasets shape: \n X_train: {X_train.shape} \n y_train:{y_train.shape}  \n X_test:{X_test.shape}  \n y_test:{y_test.shape} ")
except Exception as e:
    print(f"✗ Error loading dataset: {e}")




✓ Datasets loaded successfully!
Datasets shape: 
 X_train: (91199, 18) 
 y_train:(91199, 1)  
 X_test:(22800, 18)  
 y_test:(22800, 1) 


## 3. Categorización de la variable objetivo
 La categorización de la popularidad se realiza por género musical, teniendo en cuenta que en la descripción de los datos se evidenció que había mayor presencia de unos géneros que de otros. Por lo cual, al hacer la categorización por género se asegura que independientemente del género, la clasificación entre diferentes niveles de popularidad sea equitativa.
 
 De esta manera, la categorización se llevó a cabo teniendo en cuenta los percentiles y el género musical, de tal forma que las piezas con valor de 0 se clasifican como 'no populares' y se codifican en 0, las piezas con una popularidad menor a la mediana del género musical se clasifican como 'bajo' y se codifican en 1, las piezas con popularidad del género entre la mediana y el percentil 75 se clasifican como 'medio' y se codifican con '2', y las piezas con popularidad superior al percentil 75 del género se clasifican como altas y se codifican con 3.

De esta manera, tenemos mayor granularidad pára definir la popularidad de la canciones, teniendo en cuenta que hay mayor representacion de ciertos géneros que de otros, y que así mismo la popularidad para géneros poco comunes puede ser menor en comparación de uno más comun, pero significativa dentro del mismo género.

In [0]:
def categorize_popularity(group):
    # calculamos mediana y p75 para ese género
    median = group['popularity'].median()
    p75 = np.percentile(group['popularity'], 75)
    
    def assign_category(x):
        if x == 0:
            return 0
        elif x < median:
            return 1
        elif median <= x <= p75:
            return 2
        else:
            return 3
    
    return group['popularity'].apply(assign_category)

In [0]:
df_train=X_train.copy()
df_test=X_test.copy()



df_train['popularity']=y_train['popularity']
df_test['popularity']=y_test['popularity']
df_train['popularity_cat']=df_train.groupby('track_genre', group_keys=False).apply(categorize_popularity)
df_filtrado = df_train.groupby("track_genre").filter(lambda x: len(x) > 5)
df_test['popularity_cat']=df_test.groupby('track_genre', group_keys=False).apply(categorize_popularity)
X_train_c=df_train.drop(['popularity', 'popularity_cat'], axis=1)
X_test_c=df_test.drop(['popularity', 'popularity_cat'], axis=1)
y_train_c=df_train['popularity_cat']
y_test_c=df_test['popularity_cat']

## 4. Busqueda del mejor modelo y sus hiperparámetros.
En estos experimentos implementamos la búsqueda por grilla de los mejores hiperparámetros para los modelos AdaBoost y XGBoost, se registran los experimentos en MLflow para su posterior comparación.


In [0]:
models_and_parameters = {
    "AdaBoost": (
        AdaBoostClassifier(random_state=42),
        {"n_estimators": [50, 100, 200], "learning_rate": [0.1, 0.005]}
    ),
    "XGBoost": (
        XGBClassifier(use_label_encoder=False, eval_metric="mlogloss", random_state=42),
        {"n_estimators": [100, 200, 300], "max_depth": [2, 3, 5, 7], "learning_rate": [0.005, 0.2]}
    )
}

# ====================================
# 3. Experimentos con MLflow
# ====================================
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/song_popularity_prediction_cat_by_genre")

for model_name, (model, param_grid) in models_and_parameters.items():
    with mlflow.start_run(run_name=model_name):
        print(f"Entrenando {model_name}...")

        # Grid search
        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grid,
            cv=3,
            scoring="f1_macro",
            n_jobs=-1
        )
        grid.fit(X_train_c, y_train_c)

        best_model = grid.best_estimator_

        # Predicciones
        y_pred = best_model.predict(X_test_c)

        # Métricas
        acc = accuracy_score(y_test_c, y_pred)
        f1 = f1_score(y_test_c, y_pred, average="macro")

        print(f"{model_name} → Acc: {acc:.3f}, F1-macro: {f1:.3f}")
        print(classification_report(y_test_c, y_pred))

        # Log en MLflow
        mlflow.log_params(grid.best_params_)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1)

        # Guardar modelo
        if model_name == "XGBoost":
            mlflow.xgboost.log_model(best_model, model_name)
        else:
            mlflow.sklearn.log_model(best_model, model_name)

        # =============================
        # Importancia de variables
        # =============================
        if hasattr(best_model, "feature_importances_"):
            importances = best_model.feature_importances_
            feature_names = X_train.columns if hasattr(X_train, "columns") else [f"f{i}" for i in range(len(importances))]

            # DataFrame ordenado
            importance_df = pd.DataFrame({
                "feature": feature_names,
                "importance": importances
            }).sort_values(by="importance", ascending=False)

            # Guardar CSV en MLflow
            importance_csv = f"{model_name}_feature_importances.csv"
            importance_df.to_csv(importance_csv, index=False)
            mlflow.log_artifact(importance_csv)

            # Gráfico
            plt.figure(figsize=(8, 5))
            importance_df.set_index("feature").head(15).plot(kind="barh", legend=False)
            plt.gca().invert_yaxis()
            plt.title(f"Top 15 Importancias - {model_name}")
            plt.tight_layout()
            plt.savefig(f"{model_name}_feature_importances.png")
            mlflow.log_artifact(f"{model_name}_feature_importances.png")
            plt.close()

Entrenando AdaBoost...


/databricks/python/lib/python3.12/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warning

AdaBoost → Acc: 0.346, F1-macro: 0.164
              precision    recall  f1-score   support

           0       0.57      0.03      0.06      3236
           1       0.35      0.97      0.51      7676
           2       0.25      0.05      0.09      6530
           3       0.00      0.00      0.00      5358

    accuracy                           0.35     22800
   macro avg       0.29      0.26      0.16     22800
weighted avg       0.27      0.35      0.21     22800



2025/09/20 20:48:46 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Entrenando XGBoost...


/local_disk0/.ephemeral_nfs/envs/pythonEnv-78e73e13-2e8d-4dd9-9ee6-f62c7e1b38a1/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [20:48:48] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-78e73e13-2e8d-4dd9-9ee6-f62c7e1b38a1/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [20:48:48] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-78e73e13-2e8d-4dd9-9ee6-f62c7e1b38a1/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [20:48:49] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-78e73e13-2e8d-4dd9-9ee6-f62c7e1b38a1/lib/python3.12/site-packages/xgboost/t

XGBoost → Acc: 0.506, F1-macro: 0.518
              precision    recall  f1-score   support

           0       0.73      0.73      0.73      3236
           1       0.49      0.67      0.57      7676
           2       0.41      0.34      0.37      6530
           3       0.50      0.35      0.41      5358

    accuracy                           0.51     22800
   macro avg       0.53      0.52      0.52     22800
weighted avg       0.50      0.51      0.50     22800



2025/09/20 20:51:25 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

Verificamos que el mejor modelo sea el indicado en los experimentos en MLflow y guardamos

In [0]:
best_model

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.2, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=7, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [0]:
best_model.get_params()

{'objective': 'multi:softprob',
 'base_score': None,
 'booster': None,
 'callbacks': None,
 'colsample_bylevel': None,
 'colsample_bynode': None,
 'colsample_bytree': None,
 'device': None,
 'early_stopping_rounds': None,
 'enable_categorical': False,
 'eval_metric': 'mlogloss',
 'feature_types': None,
 'feature_weights': None,
 'gamma': None,
 'grow_policy': None,
 'importance_type': None,
 'interaction_constraints': None,
 'learning_rate': 0.2,
 'max_bin': None,
 'max_cat_threshold': None,
 'max_cat_to_onehot': None,
 'max_delta_step': None,
 'max_depth': 7,
 'max_leaves': None,
 'min_child_weight': None,
 'missing': nan,
 'monotone_constraints': None,
 'multi_strategy': None,
 'n_estimators': 300,
 'n_jobs': None,
 'num_parallel_tree': None,
 'random_state': 42,
 'reg_alpha': None,
 'reg_lambda': None,
 'sampling_method': None,
 'scale_pos_weight': None,
 'subsample': None,
 'tree_method': None,
 'validate_parameters': None,
 'verbosity': None,
 'use_label_encoder': False}

## Guardado del mejor modelo
El mejor modelo se guarda en un archivo pkl

In [0]:
import joblib

In [0]:
# guardar
joblib.dump(best_model, "modelo_popularidad.pkl")

['modelo_popularidad.pkl']